# Stage A, part 2 - the three students

Three students are trained in one loop on the pilot data of part three. Student F copies SigLIP2 FixRes, the satellite DINOv3, PE-Spatial and TerraFM-B reading Sentinel-1. Student N is F with SigLIP2 NaFlex in place of FixRes. Student C has the three C-RADIOv4 slots without the radar teacher. All three see the same batches with the same seed and the same steps, and the five teachers run once per step for all of them. SigLIP2 gives its summary only, no patch target, through the summary_only argument of the loop. The fixed values of the training (batch, steps, learning rate, schedule, clipping, PHI-S) are the defaults of TrainConfig in the code, the notebook prints them and never retypes them.

## What it does

- reads the data location and the token from the .env file. DATA_HF_REPO names the private Hugging Face repo of the pilot data, DATA_DRIVE_DIR the folder under MyDrive that holds it in pilot/. The notebook never names another project's repo or folders itself, and nothing is uploaded
- fetches the pilot files from that private repo with the read token first - the small files (chunk JSONs, infos tables, lists, statistics, manifest) 32 at once (hub.SMALL_FETCH_WORKERS), then the archives 4 files at once (hub.FETCH_WORKERS), the pair archives in chunk order - and takes the archives on Drive when the fetch fails or the repo is missing (an archive that did not arrive is read from Drive alone). The cell prints which source was used and the MB per second
- reads every pair straight out of the chunk archives into RAM in 16 threads (pilotdata.RAM_WORKERS), nothing is unpacked, with the corner rule for the 20 m and 60 m bands. Each archive is read as soon as it is on the disk, while the next ones are still being fetched (pilotdata.ChunkReader), and the cell prints the seconds this saved against fetching first and reading after. The training and held-out pairs are found by grid cell in the JSON files of the chunks before any archive is there, the infos tables of the chunks are read (built once from a tar when a chunk has none), the two aerial sets are read straight out of their archives too, while the last pairs are read. Before the arrays are made the cell checks that the RAM holds them
- reads the weight files of the five teachers once from the model cache on Drive, side by side, one thread per file (warm_weights, nothing fetched), then builds the five frozen teachers once from memory - SigLIP2 so400m FixRes at 256 px and NaFlex, the satellite DINOv3 ViT-L, PE-Spatial-L at 224 px (kept by the scene check of stage 0) and TerraFM-B through its Sentinel-1 input layer. The radar teacher gives targets for the rows whose mode is s1 or both only
- measures 20 training steps of the joint loop first and prints the split of a step (data, teachers, student F, student N, student C, rest) in seconds and shares, the images per second, the GPU memory peak and the projected hours of the joint run and of the sequential one. When the data path takes more than 15 percent of a step the notebook stops with the number and changes nothing
- trains the three students in one loop (train.train_students). Every step builds the batch once and runs the five teachers once, then F, N and C are updated one after the other, each with its own loss on its own teachers, its own optimiser (fused AdamW on the GPU), log, held-out loss and checkpoints. PHI-S is fitted once per teacher whose patches are learned (DINOv3, PE-Spatial, TerraFM-B) and saved into every folder that needs it. Checkpoint on Drive every 1,000 steps, resumable, the checkpoint before it kept as last_prev.pt
- prints the held-out losses per student and teacher, the tails of the logs and the files the scoring notebook needs
- flushes Drive at the end and releases the runtime

## Runtime

The A100 with High-RAM. The arrays take about 88 GB of RAM, part three measured it (19,988 pairs of 4.13 MB each make 82.6 GB, the aerial sets the rest), more than the standard runtime has, and the data cell stops with a message when they do not fit. The runtime disk holds the fetched archives, about 53 GB, when they come from Hugging Face. GPU memory - stage 0 put the peak of the joint loop at 29 to 31 GB by arithmetic in its two runs (the five teachers' weights 6.3 GB, the largest teacher forward 1.8 to 3.5 GB, one student step 17.3 to 17.9 GB, the other two students' states 3.5 GB). The loop is joint when at least 36 GB are free at the start (train.JOINT_MIN_GB, the arithmetic is written next to it), which holds on the 80 GB and on the 40 GB A100. With less the three students train one after the other through train.train_student with the same results, the choice and the free memory are printed. The notebook is opened from VS Code with the Colab extension.

## Run time

Estimates until the measurement cell runs, from two earlier measurement runs of the teachers - the five teachers 0.90 to 0.95 s per step, the three student steps at batch 128 0.54 to 0.58 s, so about 1.45 to 1.52 s per step and 2.5 to 2.6 h for the 6,250 steps, plus the held-out passes and the checkpoints. The data cell took 412 s in part three (the fetch 4 files at once at about 70 MB/s per file, the pairs into RAM in 16 threads at 213 MB/s, one after the other). With the reading of each archive started while the next ones are fetched it should take about 4.5 to 5 min by arithmetic (the first archives arrive after about 1 min, then the reading is the slower side), if the reading keeps its rate while the fetch runs; the cell prints the measured overlap. Loading the five teachers from the model cache on Drive took 279 s in an earlier run, the files read one after the other at 23 to 65 MB/s. Read side by side first, the reading should take about the time of the largest file, 70 to 95 s for a 4.54 GB SigLIP2 file, plus the builds from memory, by arithmetic if the five reads keep their single rates; the cell prints both. The measurement cell replaces these numbers by measured ones and keeps them on Drive. The loop prints a line every 100 steps with the step, the three losses, the seconds per step and the time left, and the held-out losses every 1,000 steps. last.pt is about 1.2 GB per student (it holds the optimiser) and final.pt about 0.4 GB, by arithmetic.

## Results

Google Drive, folder MyDrive/sar-transfer-distillation/runs_6250/ (named by the number of steps, so a run with another number of steps gets its own folder and never resumes from these checkpoints), one folder per student (student_F, student_N, student_C) with the PHI-S statistics of its teachers, the log of every step (with data_s, teacher_s and student_s next to step_s), the held-out losses, the last two checkpoints (last.pt and last_prev.pt) and final.pt. The measurement and the summary tables go to runs_6250/ too.

## How to run

Run the cells from top to bottom. After a crash on a new runtime run them again from the top. After a crash in the SAME session restart the runtime first, because the dead cell keeps the 88 GB of arrays alive (the data cell frees the arrays of an earlier run of this session before it loads again, not those of a cell that died). A student whose final.pt is on Drive is not trained again. When all three are finished the data is not loaded, the teachers are not built and nothing is measured, only the summary is printed. A student whose last checkpoint is there goes on from it, from last_prev.pt with a printed line when last.pt is missing or was cut short by a runtime that died while writing it. When the students are at different steps (a runtime that died between the saves of one checkpoint step) all go on from the smallest step, a student ahead takes its last_prev.pt at that step and computes the steps in between again, the same numbers on the CPU and the same in distribution on the GPU. Once a student has a checkpoint on Drive the measurement is not repeated, its numbers are read from measurement.json there (delete that file for a new one). A loss that is not finite stops the run with a message naming the student, the last checkpoints stay. The last cell flushes Drive and then releases the Colab machine, so the GPU stops by itself when the run is done (set END_RUNTIME = False in that cell to keep the runtime), and to use Drive afterwards run the mount cell again.

## Token

The Hugging Face token (HF_TOKEN) is read from the .env file. It opens the gated DINOv3 weights and the private pilot repo, and is never printed. The weights come from the model cache on Drive, the folder MODEL_CACHE_DRIVE_DIR of the .env; a teacher whose files are not there yet is fetched once at its pinned revision.

In [ ]:
# Mount Google Drive. The data, the results and the model cache are there.
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# Setup. Token, data location, model cache, GPU check, the fixed training values, the students, folders.
import gc
import json
import os
import shutil
import tarfile
import time
from pathlib import Path

import pandas as pd
import torch

from sartransfer import env
from sartransfer.distil import aerial, pilotdata, stage0, train
from sartransfer.distil.teachers import build_teacher, warm_weights
from sartransfer.domains import hub

start = time.perf_counter()
assert torch.cuda.is_available(), "no GPU, pick the A100 runtime with High-RAM and connect again"
DEVICE = "cuda"
MODEL_CACHE = env.model_cache_on_drive()       # before transformers or huggingface_hub load
assert MODEL_CACHE is not None, "Google Drive is not mounted, run the mount cell first"
env.load_env()
HF_TOKEN = env.require("HF_TOKEN", "the gated DINOv3 weights and the private pilot data")
assert HF_TOKEN, "no Hugging Face token was given"
DATA_DIR = env.require("DATA_DRIVE_DIR", "the folder under MyDrive with the pilot data")
# DATA_HF_REPO, the private repo of the pilot data, is read in the data cell (hub.fetch_pilot, hub.data_repo_id())

# Settings of this notebook, how the work is done. None of them is a rule.
N_MEASURE = 20                   # training steps of the measurement before the long run
DATA_SHARE_LIMIT = 0.15          # the data path above this share of a step stops the notebook
RAM_SPARE_GB = 10                # RAM that must stay free next to the arrays. The teachers pass through the
                                 # CPU while they load (6.3 GB of float32 weights, stage 0), Python and torch
                                 # need some more. A round number above that, not measured

# The teachers and the students
TEACHER_NAMES = ("siglip2-so400m-256", "siglip2-so400m-naflex", "dinov3-l-sat", "pe-spatial-l",
                 "terrafm-b-radar")           # the five models of the four slots
SUMMARY_ONLY = ("siglip2-so400m-256", "siglip2-so400m-naflex")   # SigLIP2 gives its summary target only
STUDENTS = {"F": ["siglip2-so400m-256", "dinov3-l-sat", "pe-spatial-l", "terrafm-b-radar"],
            "N": ["siglip2-so400m-naflex", "dinov3-l-sat", "pe-spatial-l", "terrafm-b-radar"],
            "C": ["siglip2-so400m-256", "dinov3-l-sat", "pe-spatial-l"]}

# The fixed values are the defaults of the code. Printed, never retyped.
CFG = train.TrainConfig()
print(f"steps {CFG.steps:,}, batch {CFG.batch} = {CFG.n_majortom} Major-TOM + {CFG.n_aerial} aerial, "
      f"crop {CFG.crop} px, peak lr {CFG.lr:g}, weight decay {CFG.weight_decay:g}, warm-up {CFG.warmup} "
      f"steps, cosine to 0, clip {CFG.clip:g}, bf16 {CFG.bf16}, seed {CFG.seed}")
print(f"checkpoint every {CFG.checkpoint_every:,} steps, held-out loss every {CFG.held_out_every:,}, progress line "
      f"every {CFG.progress_every}, PHI-S on {CFG.phis_samples:,} samples per teacher, "
      f"student {CFG.student or 'ViT-B/16 at 256 px'}")
for key, names in STUDENTS.items():
    print(f"student {key}  " + ", ".join(f"{name} (summary only)" if name in SUMMARY_ONLY else name
                                         for name in names))

CONTENT, MYDRIVE = (Path(folder) for folder in stage0.WORK_FOLDERS)
PILOT = MYDRIVE / DATA_DIR / "pilot"                    # what part three's data notebook wrote, read in place
DRIVE_MT, DRIVE_AERIAL = PILOT / "majortom", PILOT / "aerial"
RUNS = MYDRIVE / "sar-transfer-distillation" / f"runs_{CFG.steps}"   # the students, on Drive, one folder per schedule
OUT = {key: RUNS / f"student_{key}" for key in STUDENTS}
SPECS = [(OUT[key].name, names) for key, names in STUDENTS.items()]   # what train.train_students takes
WORK = CONTENT / "sar-transfer-distillation" / "distil"        # runtime disk, lost with the runtime
HUB_DIR = WORK / "hub"                                  # the files fetched from the private repo
for folder, why in ((DRIVE_MT, "the pair archives"), (DRIVE_AERIAL, "the aerial archives")):
    print(f"{'on Drive     ' if folder.is_dir() else 'NOT on Drive '} {folder.relative_to(MYDRIVE / DATA_DIR).as_posix()} "
          f"of DATA_DRIVE_DIR ({why})")
for folder in (*OUT.values(), WORK):
    folder.mkdir(parents=True, exist_ok=True)

TIMES = {}                       # seconds of every step of this run, by name


def num(value, digits=2):
    """A number as text, "not known" for None and nan."""
    if value is None or value != value:
        return "not known"
    return f"{value:,.{digits}f}"


def show(table):
    """Print a whole table."""
    print(table.to_string(index=False))


def hide(text):
    """A text without the tokens."""
    out = str(text)
    for secret in (HF_TOKEN, os.environ.get("HF_WRITE_TOKEN")):
        if secret:
            out = out.replace(secret, "<token>")
    return out


def took(since, name):
    """The time of a cell as text, recorded under `name` for the summary."""
    TIMES[name] = time.perf_counter() - since
    return f"cell time {TIMES[name]:.1f} s"


runtime = stage0.runtime_report([WORK, RUNS])
print(f"GPU     {runtime['gpu']}, {num(runtime['gpu_memory_gb'], 1)} GB, "
      f"free {num(runtime['gpu_free_gb'], 1)} GB")
print(f"RAM     {num(runtime['ram_gb'], 1)} GB, free {num(runtime['ram_free_gb'], 1)} GB, "
      f"{runtime['cpus']} CPUs")
for folder, disk in runtime["disk"].items():
    print(f"disk    {folder}, free {num(disk['free_gb'], 1)} of {num(disk['total_gb'], 1)} GB")
print(f"students go to  {RUNS}")
print(f"work folder     {WORK}")
stage0.save_report(RUNS, runtime=runtime)
print(took(start, "setup"))


In [ ]:
# Data. The pilot files from the private Hugging Face repo of DATA_HF_REPO, else the archives on Drive under
# DATA_DRIVE_DIR, the chunk JSONs, the held-out lists, every pair straight from the archives into RAM (Sentinel-2 on
# the 10 m grid with the corner rule, nothing unpacked), each archive read while the next ones are still being
# fetched, the two aerial sets straight from their archives, the statistics. Every fetch and read prints its MB per
# second, the source used and the seconds the overlap saved are printed. Nothing is loaded when every student is
# finished on Drive.
start = time.perf_counter()
for _name in ("arrays", "data", "ram", "ram_train", "ram_held", "sets", "reader"):   # a second run in this session
    globals().pop(_name, None)              # frees the 88 GB of the first run before the new load
gc.collect()
AERIAL_SETS = ("distil_train", "distil_heldout")
if all((out / train.FINAL).is_file() for out in OUT.values()):
    data = None
    print(f"every student is finished on Drive ({train.FINAL}), the data is not loaded. Delete a {train.FINAL} "
          "there to train that student again")
else:
    # the source. First the small files of the private repo (the chunk JSONs and their infos tables, the two lists,
    # the statistics, the aerial manifest) land on the runtime disk, hub.SMALL_FETCH_WORKERS at once (the reason is
    # written there), complete ones are skipped. Then the archives on Drive when that fails or the repo is missing.
    # Either way the archives are read in place, nothing of them is unpacked or copied
    SMALL = ("majortom/chunk_*.json", "majortom/chunk_*_infos.csv", f"majortom/{pilotdata.TRAIN_FILE}",
             f"majortom/{pilotdata.HELD_OUT_FILE}", "majortom/channel_stats.json", "aerial/manifest.json")
    ARCHIVES = ("majortom/chunk_*.tar", *(f"aerial/{name}.tar" for name in AERIAL_SETS))
    clock = time.perf_counter()
    try:
        fetched = hub.fetch_pilot(HUB_DIR, token=HF_TOKEN, want=SMALL, workers=hub.SMALL_FETCH_WORKERS)
        SOURCE_MT, SOURCE_AERIAL = HUB_DIR / "majortom", HUB_DIR / "aerial"
        print(f"pilot data from the private Hugging Face repo of DATA_HF_REPO, {len(fetched)} small files in "
              f"{time.perf_counter() - clock:.1f} s, the archives follow while the pairs are read")
    except Exception as err:
        SOURCE_MT, SOURCE_AERIAL = DRIVE_MT, DRIVE_AERIAL
        print(f"pilot data from the archives on Drive, not from Hugging Face ({type(err).__name__}, "
              f"{hide(err)[:200]})")
        for folder, why in ((DRIVE_MT, "the pair archives"), (DRIVE_AERIAL, "the aerial archives")):
            assert folder.is_dir(), (f"{folder.relative_to(MYDRIVE / DATA_DIR).as_posix()} of DATA_DRIVE_DIR is not "
                                     f"on Drive either ({why}), the pilot data of part three is needed")

    # the pairs, chunk by chunk. The JSON of a chunk names its pair files and the bytes of its archive, so every pair
    # of the two lists is found by its grid cell before any archive is on the disk (pilotdata.pair_locations)
    chunks = sorted(SOURCE_MT.glob("chunk_*.json"), key=lambda p: int(p.stem.split("_")[1]))
    assert chunks, "no chunk archive in the source, the pilot data of part three is needed"
    numbers, archive_bytes = [], {}
    for info_path in chunks:
        info = json.loads(info_path.read_text(encoding="utf-8"))
        k = int(info["chunk"])
        numbers.append(k)
        archive_bytes[k] = int(info["bytes"])
        print(f"chunk {k}  {len(info['files']):,} pairs in chunk_{k}.tar, read in place")
    where = pilotdata.pair_locations(SOURCE_MT, numbers)
    print(f"{len(where):,} pairs in {len(numbers)} archives, {sum(archive_bytes.values()) / stage0.GB:.1f} GB")
    lists = {}
    for name, file in (("train", pilotdata.TRAIN_FILE), ("held_out", pilotdata.HELD_OUT_FILE)):
        path = SOURCE_MT / file
        assert path.is_file(), f"{file} is missing in the source"
        cells = path.read_text(encoding="utf-8").split()
        missing = [c for c in cells if c not in where]
        assert not missing, (f"{len(missing)} cells of {file} have no pair file in the archives, for example "
                             f"{missing[:3]}")
        lists[name] = [where[c] for c in cells]
        print(f"{name:<9} {len(cells):,} pairs from {file}")
    assert not set(lists["train"]) & set(lists["held_out"]), "a pair is in both lists"

    STATS = SOURCE_MT / "channel_stats.json"
    assert STATS.is_file(), f"{STATS.name} is missing in the source"
    stats = json.loads(STATS.read_text(encoding="utf-8"))
    print(f"channel statistics from {STATS.name}, {stats['n']:,} pairs, corner rule {stats['corner_rule']}")
    MANIFEST = SOURCE_AERIAL / "manifest.json"
    assert MANIFEST.is_file(), f"the aerial {MANIFEST.name} is missing in the source"
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))

    # the RAM, checked before anything is allocated - pilotdata.PAIR_BYTES per pair, 8 bit per aerial value
    n_pairs = len(lists["train"]) + len(lists["held_out"])
    n_patches = sum(int(manifest["sets"][name]["count"]) for name in AERIAL_SETS)
    need_gb = (n_pairs * pilotdata.PAIR_BYTES + n_patches * len(manifest["bands"]) * manifest["out_px"] ** 2) / stage0.GB
    free_gb = stage0.runtime_report([WORK])["ram_free_gb"]
    print(f"the arrays will take {need_gb:.1f} GB of RAM ({n_pairs:,} pairs, {n_patches:,} aerial patches), "
          f"{free_gb:.1f} GB are free, {RAM_SPARE_GB} GB must stay free")
    assert need_gb + RAM_SPARE_GB <= free_gb, (
        f"the arrays need {need_gb:.1f} GB and {RAM_SPARE_GB} GB must stay free, only {free_gb:.1f} GB of RAM are "
        "free. Pick the A100 runtime with High-RAM and run the notebook again from the top")

    # the pairs into RAM while the archives arrive. pilotdata.ChunkReader reads one archive at a time in its own
    # thread, pilotdata.RAM_WORKERS threads on it (the reason is written there), into arrays made once here, and
    # reads the infos table of the archive or builds it once from the tar. hub.fetch_pilot hands every archive over
    # as soon as it is whole on the disk, hub.FETCH_WORKERS files at once (the reason is written there), the pair
    # archives in chunk order. An archive that did not arrive (the fetch failed, or the source is Drive) is read
    # from Drive in place. The aerial sets are read here while the reader is still at the pairs
    reader = pilotdata.ChunkReader({"train": lists["train"], "held_out": lists["held_out"]}, corner_rule=True)
    try:
        ARRIVED = set()

        def arrived(name, path):              # the fetch calls this for every archive that is whole on the disk
            ARRIVED.add(name)
            if name.startswith("majortom/"):
                reader.arrived(path)

        clock = time.perf_counter()
        if SOURCE_MT == HUB_DIR / "majortom":
            try:
                hub.fetch_pilot(HUB_DIR, token=HF_TOKEN, want=ARCHIVES, on_file=arrived)
            except Exception as err:
                print(f"the archives that did not arrive are read from Drive ({type(err).__name__}, "
                      f"{hide(err)[:200]})")
        fetch_s = time.perf_counter() - clock
        for k in numbers:
            if f"majortom/chunk_{k}.tar" not in ARRIVED:
                archive = DRIVE_MT / f"chunk_{k}.tar"
                assert archive.is_file() and archive.stat().st_size == archive_bytes[k], (
                    f"{archive.name} is missing or cut, fetch it again (or rebuild it with notebook 01)")
                reader.arrived(archive)

        # the aerial sets, the npz members straight out of each set's tar (a file object per member, nothing
        # unpacked to the disk), the arrays made once at full size as aerial.load_set does
        sets, aerial_s = {}, 0.0
        for name in AERIAL_SETS:
            archive = (HUB_DIR / "aerial" if f"aerial/{name}.tar" in ARRIVED else DRIVE_AERIAL) / f"{name}.tar"
            assert archive.is_file(), f"{archive.name} is missing in the source"
            clock_set = time.perf_counter()
            try:
                sets[name] = aerial.load_set_from_tar(archive, name, manifest["sets"][name]["files"])
            except FileNotFoundError as err:          # a file of the manifest is not in the tar
                raise AssertionError(hide(err)) from None
            except (tarfile.TarError, EOFError, OSError) as err:
                raise RuntimeError(f"{archive.name} cannot be read, {type(err).__name__}, {hide(err)}. "
                                   "Fetch it again (or rebuild it with notebook 01)") from None
            seconds = time.perf_counter() - clock_set
            aerial_s += seconds
            mb = archive.stat().st_size / stage0.MB
            print(f"aerial {name:<15} read in place from {archive.name}  {sets[name]['images'].shape}, "
                  f"{mb:,.0f} MB in {seconds:.1f} s, {mb / max(seconds, 1e-9):,.0f} MB per second")
        ram = reader.finish()
    except BaseException:
        reader.stop()                             # no reading goes on and its arrays leave RAM
        raise
    ram_train, ram_held = ram["train"], ram["held_out"]
    print(f"pairs in RAM, {reader.read_mb:,.0f} MB read from the archives in {reader.read_s:.0f} s, "
          f"{reader.read_mb / max(reader.read_s, 1e-9):,.0f} MB per second, training {ram_train['s2'].shape}, held-out "
          f"{ram_held['s2'].shape}, corner rule applied {ram_train['corner_rule']}, {pilotdata.RAM_WORKERS} threads")
    wall, read_s = time.perf_counter() - clock, reader.read_s + aerial_s
    print(f"fetch and read side by side {wall:.0f} s, one after the other {fetch_s + read_s:.0f} s ({fetch_s:.0f} s "
          f"fetching the archives, {read_s:.0f} s reading the pairs and the aerial sets), {fetch_s + read_s - wall:.0f} s "
          "saved")

    data = train.TrainData(ram_train["s1"], ram_train["s2"], sets["distil_train"]["images"], stats,
                           ram_held["s1"], ram_held["s2"], sets["distil_heldout"]["images"])
    data.check(CFG.crop)
    arrays = (data.s1, data.s2, data.aerial, data.held_s1, data.held_s2, data.held_aerial)
    in_ram = sum(x.nbytes for x in arrays) / stage0.GB
    print(f"\nthe arrays take {in_ram:.1f} GB, free RAM now {num(stage0.runtime_report([WORK])['ram_free_gb'], 1)} GB")
print(took(start, "data"))

In [ ]:
# Teachers. The five, frozen, built once here for the measurement and the long run.
# SigLIP2 FixRes and NaFlex, DINOv3 SAT and PE-Spatial read the Sentinel-2 thumbnail and the aerial RGB with
# their own readers, TerraFM-B reads the Sentinel-1 crop of the rows whose mode is s1 or both. SigLIP2 is marked
# summary only. Not built when every student is finished on Drive. Their weight files are read once from the
# model cache on Drive first, side by side (warm_weights), so the builds read them from the page cache instead of
# one file after the other over the Drive mount (279 s for the five in an earlier run).
start = time.perf_counter()
teachers = targets = t = None               # a second run in this session frees the teachers of the first
gc.collect()
torch.cuda.empty_cache()
if all((out / train.FINAL).is_file() for out in OUT.values()):
    print("every student is finished on Drive, the teachers are not built")
else:
    # one thread per weight file, teachers.WARM_WORKERS at once (the reason is written there), no network. A file
    # that is not in the cache is named and left to its build, which fetches it as before
    warmed = warm_weights(TEACHER_NAMES)
    print(f"weights read ahead, {len(warmed['files'])} files, {warmed['bytes'] / stage0.GB:.2f} GB in "
          f"{warmed['seconds']:.1f} s, {warmed['mb_s']:,.0f} MB per second with {warmed['workers']} threads, "
          f"{len(warmed['missing'])} not in the cache, {len(warmed['failed'])} not readable")
    teachers, n_params = [], 0
    for name in TEACHER_NAMES:
        clock = time.perf_counter()
        t = build_teacher(name, device=DEVICE, token=HF_TOKEN)
        teachers.append(t)
        n_params += t.n_params()
        print(f"{t.name:<22} loaded in {time.perf_counter() - clock:.1f} s, {t.n_params():,} parameters, rule "
              f"{t.rule}, input {t.input_size} px, grid {t.grid} x {t.grid}, width {t.dim}")
    targets = train.as_targets(teachers, summary_only=SUMMARY_ONLY)
    for t in targets:
        print(f"{t.name:<22} reads {'Sentinel-1' if t.radar is not None else 'Sentinel-2'}, aerial target "
              f"{'yes' if t.aerial is not None else 'no'}, {'summary only' if t.summary_only else 'summary and patches'}")
    t = None
    seconds = time.perf_counter() - start
    print(f"{len(teachers)} teachers, {n_params / 1e6:,.0f} M parameters in {seconds:.1f} s ({warmed['seconds']:.1f} s "
          f"reading ahead, {seconds - warmed['seconds']:.1f} s building), {n_params / 1e6 / max(seconds, 1e-9):,.0f} M "
          "parameters per second")
print(took(start, "teachers"))


In [ ]:
# Measurement. 20 training steps of the joint loop (all three students, the teachers once per step) in a scratch
# folder on the runtime disk before the long run. The split of a step (data, teachers, every student, rest) in
# seconds and shares, the images per second, the GPU memory, the projected hours of the joint run and of the
# sequential one. A data path above the limit stops the notebook with an advice. Once a student has a checkpoint
# on Drive (last.pt, or last_prev.pt alone) the long run has begun and the measurement on Drive is kept, not taken
# again. On a GPU with too little free memory train_students measures the sequential way instead (the choice is
# printed) and the other way is projected.
start = time.perf_counter()
MEASUREMENT = RUNS / "measurement.json"
if all((out / train.FINAL).is_file() for out in OUT.values()):
    print("every student is finished on Drive, no measurement is needed")
    measured = None
elif any((out / f).is_file() for out in OUT.values() for f in (train.LAST, train.PREV)) and MEASUREMENT.is_file():
    measured = json.loads(MEASUREMENT.read_text(encoding="utf-8"))
    print("a student has a checkpoint on Drive, the long run has begun. The measurement is not repeated, its "
          f"numbers come from {MEASUREMENT.name}. Delete that file for a new one")
    print(f"seconds per step   {measured['step_s']:.3f} ({measured['mode']}), of it the data path "
          f"{measured['data_s']:.3f} s, share {measured['data_share']:.1%}, {measured['images_per_s']:,.0f} images per "
          f"second, GPU memory peak {measured['gpu_gb']:.1f} GB, projected {measured['hours_projected']:.2f} h on "
          f"{measured['device']}")
else:
    scratch = WORK / "measure"
    shutil.rmtree(scratch, ignore_errors=True)
    results = train.train_students(SPECS, CFG, data, targets, scratch, resume=False, device=DEVICE,
                                   stop_after=N_MEASURE, summary_only=SUMMARY_ONLY)
    mode = results[SPECS[0][0]]["mode"]
    logs = {key: pd.read_csv(results[out.name]["log"]) for key, out in OUT.items()}
    first = next(iter(OUT))                   # student F, in the joint loop its log holds the times of every step
    n_steps = len(logs[first])
    warm = min(5, max(n_steps - 1, 0))        # the first steps warm up, they are not counted
    steady = {key: log.iloc[warm:] for key, log in logs.items()}

    def mean(key, column):
        return float(steady[key][column].mean())

    student_s = {key: mean(key, "student_s") for key in OUT}
    if mode == "joint":                       # one step holds the data, the teachers once and every update
        data_s, teacher_s, step_s = mean(first, "data_s"), mean(first, "teacher_s"), mean(first, "step_s")
        rest_s = step_s - data_s - teacher_s - sum(student_s.values())
        one_step = step_s
        hours_joint = CFG.steps * step_s / 3600
        # one after the other pays the data, the teachers and the rest once per student. A student alone runs its
        # own teachers only (four or three of the five), so the real sequential run is a little shorter than this
        hours_seq = CFG.steps * (step_s + (len(OUT) - 1) * (data_s + teacher_s + rest_s)) / 3600
    else:                                     # one run per student, a step holds the data, its teachers, one update
        alone = {key: mean(key, "step_s") for key in OUT}
        slowest = max(alone, key=alone.get)
        data_s, teacher_s, one_step = mean(slowest, "data_s"), mean(slowest, "teacher_s"), alone[slowest]
        rest_s = one_step - data_s - teacher_s - student_s[slowest]
        step_s = sum(alone.values())
        hours_seq = CFG.steps * step_s / 3600
        # the step of the slowest student plus the updates of the others. The teachers of the others that it
        # lacks are not counted, so the real joint run is a little longer than this
        hours_joint = CFG.steps * (one_step + sum(s for key, s in student_s.items() if key != slowest)) / 3600
    measured = {"mode": mode, "steps": n_steps, "steps_counted": int(len(steady[first])), "step_s": step_s,
                "one_step_s": one_step, "data_s": data_s, "teacher_s": teacher_s, "student_s": student_s,
                "rest_s": rest_s, "data_share": data_s / one_step, "images_per_s": CFG.batch / one_step,
                "gpu_gb": max(float(log["gpu_gb"].max()) for log in logs.values()),
                "hours_projected": hours_joint if mode == "joint" else hours_seq,
                "hours_joint": hours_joint, "hours_sequential": hours_seq,
                "loss_first": {key: float(log["loss"].iloc[0]) for key, log in logs.items()},
                "loss_last": {key: float(log["loss"].iloc[-1]) for key, log in logs.items()},
                "device": runtime["gpu"], "batch": CFG.batch, "students": STUDENTS}
    print(f"{n_steps} steps measured ({mode}), the last {measured['steps_counted']} counted")
    print(f"seconds per step   {step_s:.3f}" + ("" if mode == "joint" else
          f" for the {len(OUT)} students one after the other ({' + '.join(f'{alone[key]:.3f}' for key in OUT)})"))
    print("split of a step    " + ", ".join(
        f"{what} {s:.3f} s ({s / one_step:.0%})" for what, s in
        (("data", data_s), ("teachers", teacher_s), *((f"student {key}", student_s[key]) for key in OUT),
         ("rest", rest_s))))
    print(f"images per second  {measured['images_per_s']:,.0f}, GPU memory peak {measured['gpu_gb']:.1f} GB")
    print(f"projected          joint {hours_joint:.2f} h ({'measured' if mode == 'joint' else 'estimated'}), "
          f"one after the other {hours_seq:.2f} h ({'estimated' if mode == 'joint' else 'measured'}), "
          f"for {CFG.steps:,} steps without the held-out losses")
    print("loss               " + ", ".join(
        f"{key} {measured['loss_first'][key]:.4f} at the first step and {measured['loss_last'][key]:.4f} "
        "at the last" for key in OUT))
    stage0.save_report(RUNS, measurement=measured)
    shutil.rmtree(scratch, ignore_errors=True)
    del results, logs, steady                 # the measured students and heads leave the GPU
    gc.collect()
    torch.cuda.empty_cache()
    if measured["data_share"] > DATA_SHARE_LIMIT:
        print(f"\nADVICE  the data path takes {measured['data_share']:.0%} of a step, more than "
              f"{DATA_SHARE_LIMIT:.0%}, although the batch of the next step is built in a thread while "
              "the GPU works on the current one (train.Prefetch). So the crop loop on the CPU is slower "
              "than a GPU step and the GPU waits for it. The plan fixes the batch, fewer crops per step "
              "are not an option. Report the numbers above. Nothing was changed automatically")
        raise RuntimeError("the data path is too slow, see the advice above")
    print(f"\nthe data path takes {measured['data_share']:.0%} of a step, under {DATA_SHARE_LIMIT:.0%}, "
          "the long run can start")
print(took(start, "measure"))


In [ ]:
# Students. F, N and C in one loop (train.train_students) on the same batches with the same seed and steps, the
# five teachers once per step for all three, SigLIP2 summary only. The loop is used when at least
# train.JOINT_MIN_GB GB of GPU memory are free, else the three train one after the other with the same results,
# the choice and the free memory are printed by the call. PHI-S is fitted once per teacher and kept on Drive in
# every folder that needs it. Checkpoints on Drive, resumable, all go on from the smallest step when they differ.
# A finished student is not trained again.
start = time.perf_counter()
todo = []
for key, names in STUDENTS.items():
    out = OUT[key]
    for name in names:
        if name in SUMMARY_ONLY:                       # its patches are not learned, it has no PHI-S
            continue
        found = train.phis_path(out, name).is_file()
        print(f"{key}  phis {name}  "
              f"{'found on Drive' if found else 'not on Drive, fitted now or copied from another folder'}")
    if (out / train.FINAL).is_file():
        print(f"{key}  {train.FINAL} found on Drive, not trained again. Delete it, {train.LAST} and "
              f"{train.PREV} there to train again")
        continue
    for checkpoint in (train.LAST, train.PREV):        # last.pt, else the one before it
        if (out / checkpoint).is_file():
            print(f"{key}  {checkpoint} found on Drive, the run goes on from it")
            break
    todo.append((out.name, names))
results = {}
if todo:
    trained = train.train_students(todo, CFG, data, targets, RUNS, resume=True, device=DEVICE,
                                   summary_only=SUMMARY_ONLY)
    for key, out in OUT.items():
        if out.name in trained:
            result = trained[out.name]
            print(f"{key}  step {result['step']:,} of {CFG.steps:,}, final {result['final']}")
            # the step, the held-out losses and the paths are kept. The students, the heads and the PHI-S
            # transforms are not, so their GPU memory is freed. Notebook 03 reads final.pt
            results[key] = {k: v for k, v in result.items() if k not in ("student", "heads", "phis")}
    del trained, result
    gc.collect()
    torch.cuda.empty_cache()
    log = pd.read_csv(results[next(iter(results))]["log"])
    print(f"the run so far, {len(log):,} steps, {log['step_s'].mean():.3f} s per step, "
          f"{log['img_s'].mean():,.0f} images per second, GPU memory peak {log['gpu_gb'].max():.1f} GB")
else:
    print("every student is finished on Drive, nothing to train")
result_f, result_n, result_c = (results.get(key) for key in OUT)
print(took(start, "students"))


In [ ]:
# Summary. The held-out losses per student and teacher, the tails of the logs with the split of a step, the
# files on Drive, the paths notebook 03 reads.
start = time.perf_counter()
pd.set_option("display.width", 250)
final_paths, last_held = {}, {}
for key, out in OUT.items():
    print(f"\n---- student {key}  {out.name}, teachers {', '.join(STUDENTS[key])}")
    held = out / train.HELD
    rows = pd.read_csv(held) if held.is_file() else None
    if rows is not None and len(rows):
        print(f"held-out loss, {len(rows)} rows, the last ones")
        show(rows.tail(3))
        last_held[key] = rows.iloc[-1]
    else:
        print("no held-out loss yet")
    log = out / train.LOG
    if log.is_file():
        rows = pd.read_csv(log)
        print(f"log, {len(rows):,} steps, the last rows")
        show(rows.tail(3)[["step", "lr", "loss", "grad_norm", "step_s", "data_s", "teacher_s", "student_s",
                           "img_s", "gpu_gb"]])
    final = out / train.FINAL
    final_paths[key] = str(final) if final.is_file() else None
    print(f"{train.FINAL}  {'on Drive' if final.is_file() else 'NOT on Drive, the student is not finished'}")
    for path in sorted(out.iterdir()):
        print(f"  {path.name:<40} {path.stat().st_size / stage0.MB:>10.1f} MB")
# the last held-out row of every student side by side, one row per teacher part, - where it is not that student's
parts = list(dict.fromkeys(column for row in last_held.values() for column in row.index))
held_table = pd.DataFrame({key: row.reindex(parts) for key, row in last_held.items()}, index=parts)
if last_held:
    print("\nheld-out losses at the last held-out step, per teacher part (rows) and student (columns)")
    print(held_table.to_string(na_rep="-"))
    stage0.save_report(RUNS, held_out_last=held_table.rename_axis("part").reset_index())
stage0.save_report(RUNS, final_paths=final_paths, times=TIMES)
print("\nnotebook 03 reads")
for key, path in final_paths.items():
    print(f"  student {key}  {path or 'not finished'}")
print(took(start, "summary"))


In [ ]:
# Flush Drive and release the runtime. The run is over when this cell prints, Drive then holds
# everything that was written. With END_RUNTIME the Colab machine is released right after, so
# the GPU stops and no compute is spent waiting - google.colab.runtime.unassign asks Colab from
# the machine itself (an HTTP request to the runtime service, read in the colabtools source on
# 2026-10-04), so it works from VS Code as well as from the browser. A cell that fails earlier
# stops the run before this one, so a runtime with an error stays up to be looked at. Set
# END_RUNTIME = False to keep the runtime; run the mount cell again to use Drive afterwards.
from google.colab import drive, runtime

END_RUNTIME = True               # release the machine after the flush, the outputs stay in the notebook

drive.flush_and_unmount()
print("Drive flushed and unmounted, the run is over"
      + (", the runtime is released now" if END_RUNTIME else ", the runtime stays up"), flush=True)
if END_RUNTIME:
    runtime.unassign()
